# AI-Based Intrusion Detection System

This notebook implements a basic anomaly-based intrusion detection system using **K-Means clustering**.

The workflow includes:
- Loading the NSL-KDD dataset when available
- Falling back to a small synthetic dataset if the dataset file is not found
- Preprocessing and normalizing network traffic features
- Training a K-Means model
- Detecting anomalies using distance-based thresholding
- Simulating incoming network traffic and generating anomaly alerts
- Saving the trained model and parameters

In [ ]:
# If you run this notebook in Google Colab, uncomment the line below if needed.
# !pip install pandas numpy scikit-learn matplotlib joblib -q

import os
import json
import urllib.request
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")

## 1. Dataset Loading

The notebook first tries to load `KDDTrain+.txt` from the current directory. If the file is not available, it tries to download it from a public source. If that also fails, it creates a synthetic traffic-like dataset so the notebook can still run end-to-end.

In [ ]:
NSL_KDD_COLUMNS = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

DATASET_FILE = Path('KDDTrain+.txt')
DATASET_URL = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt'

def create_synthetic_dataset(n_normal=600, n_attack=180):
    # Create a small traffic-like dataset for demonstration when NSL-KDD is unavailable.
    normal = pd.DataFrame({
        'duration': np.random.exponential(1.0, n_normal),
        'src_bytes': np.random.normal(350, 80, n_normal).clip(10, None),
        'dst_bytes': np.random.normal(450, 100, n_normal).clip(10, None),
        'count': np.random.poisson(12, n_normal),
        'srv_count': np.random.poisson(10, n_normal),
        'serror_rate': np.random.uniform(0.00, 0.15, n_normal),
        'srv_serror_rate': np.random.uniform(0.00, 0.15, n_normal),
        'dst_host_count': np.random.poisson(80, n_normal),
        'dst_host_srv_count': np.random.poisson(70, n_normal),
        'label': 'normal'
    })

    attack = pd.DataFrame({
        'duration': np.random.exponential(4.0, n_attack),
        'src_bytes': np.random.normal(1500, 400, n_attack).clip(50, None),
        'dst_bytes': np.random.normal(120, 50, n_attack).clip(1, None),
        'count': np.random.poisson(80, n_attack),
        'srv_count': np.random.poisson(65, n_attack),
        'serror_rate': np.random.uniform(0.45, 1.00, n_attack),
        'srv_serror_rate': np.random.uniform(0.45, 1.00, n_attack),
        'dst_host_count': np.random.poisson(200, n_attack),
        'dst_host_srv_count': np.random.poisson(25, n_attack),
        'label': 'attack'
    })
    return pd.concat([normal, attack], ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

def load_dataset():
    if DATASET_FILE.exists():
        print(f"Loading local dataset: {DATASET_FILE}")
        return pd.read_csv(DATASET_FILE, names=NSL_KDD_COLUMNS)

    try:
        print("Local dataset not found. Trying to download NSL-KDD dataset...")
        with urllib.request.urlopen(DATASET_URL, timeout=10) as response:
            DATASET_FILE.write_bytes(response.read())
        print(f"Downloaded dataset: {DATASET_FILE}")
        return pd.read_csv(DATASET_FILE, names=NSL_KDD_COLUMNS)
    except Exception as error:
        print("Dataset could not be downloaded. Synthetic dataset will be used instead.")
        print(f"Reason: {error}")
        return create_synthetic_dataset()

raw_df = load_dataset()
print("Dataset shape:", raw_df.shape)
raw_df.head()

## 2. Feature Selection and Preprocessing

Only numeric traffic features are used for K-Means. The model is trained mainly on normal traffic, then distances to cluster centers are used as anomaly scores.

In [ ]:
CANDIDATE_FEATURES = [
    'duration', 'src_bytes', 'dst_bytes', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'dst_host_count', 'dst_host_srv_count'
]

selected_features = [col for col in CANDIDATE_FEATURES if col in raw_df.columns]
if not selected_features:
    raise ValueError("No numeric features were found for model training.")

# Convert selected columns to numeric and fill missing values.
df = raw_df.copy()
for col in selected_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[selected_features] = df[selected_features].fillna(0)

# Create binary labels when the label column exists.
if 'label' in df.columns:
    df['is_attack'] = df['label'].apply(lambda x: 0 if str(x).lower() == 'normal' else 1)
else:
    df['is_attack'] = np.nan

print("Selected features:", selected_features)
print(df[selected_features + ['label']].head() if 'label' in df.columns else df[selected_features].head())

## 3. K-Means Training

The model learns common traffic patterns. A high distance from the nearest cluster center is interpreted as an anomaly.

In [ ]:
# Train on normal traffic when labels are available.
if 'is_attack' in df.columns and df['is_attack'].notna().any() and (df['is_attack'] == 0).sum() > 20:
    train_df = df[df['is_attack'] == 0].copy()
else:
    train_df = df.copy()

X_train = train_df[selected_features]
X_all = df[selected_features]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_all_scaled = scaler.transform(X_all)

kmeans_model = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
kmeans_model.fit(X_train_scaled)

# Distance to nearest cluster center = anomaly score.
train_distances = kmeans_model.transform(X_train_scaled).min(axis=1)
all_distances = kmeans_model.transform(X_all_scaled).min(axis=1)

threshold = float(np.percentile(train_distances, 95))
df['anomaly_score'] = all_distances
df['predicted_anomaly'] = (df['anomaly_score'] > threshold).astype(int)

print(f"Training rows: {len(train_df)}")
print(f"Threshold: {threshold:.4f}")
print(df[['anomaly_score', 'predicted_anomaly']].head())

## 4. Evaluation

If labels are available, the notebook calculates basic classification metrics. In an unsupervised anomaly detection setup, these metrics are mainly used for evaluation, not for model training.

In [ ]:
if df['is_attack'].notna().any():
    y_true = df['is_attack'].astype(int)
    y_pred = df['predicted_anomaly'].astype(int)

    print("Confusion Matrix")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))
else:
    print("Labels are not available. Evaluation metrics were skipped.")

plt.figure(figsize=(8, 4))
plt.hist(df['anomaly_score'], bins=40)
plt.axvline(threshold, linestyle='--', label='Threshold')
plt.title('Anomaly Score Distribution')
plt.xlabel('Distance to nearest cluster center')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Simulated Real-Time Detection

This section simulates incoming network records and applies the trained model to generate anomaly alerts.

In [ ]:
def detect_batch(batch_df, model, scaler, features, threshold):
    # Detect anomalies in a batch of incoming traffic records.
    batch = batch_df.copy()
    for col in features:
        if col not in batch.columns:
            batch[col] = 0
        batch[col] = pd.to_numeric(batch[col], errors='coerce').fillna(0)

    X = scaler.transform(batch[features])
    distances = model.transform(X).min(axis=1)
    batch['anomaly_score'] = distances
    batch['is_anomaly'] = distances > threshold
    return batch

sample_traffic = pd.DataFrame([
    {'duration': 0.4, 'src_bytes': 320, 'dst_bytes': 480, 'count': 10, 'srv_count': 9, 'serror_rate': 0.05, 'srv_serror_rate': 0.03, 'dst_host_count': 75, 'dst_host_srv_count': 70},
    {'duration': 8.0, 'src_bytes': 2600, 'dst_bytes': 40, 'count': 140, 'srv_count': 100, 'serror_rate': 0.92, 'srv_serror_rate': 0.88, 'dst_host_count': 230, 'dst_host_srv_count': 15},
])

result = detect_batch(sample_traffic, kmeans_model, scaler, selected_features, threshold)
result[['anomaly_score', 'is_anomaly']]

## 6. Save Model Artifacts

The trained model, scaler, and model parameters are saved so they can be reused later.

In [ ]:
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(kmeans_model, ARTIFACT_DIR / 'kmeans_model.pkl')
joblib.dump(scaler, ARTIFACT_DIR / 'scaler.pkl')

model_parameters = {
    'selected_features': selected_features,
    'threshold': threshold,
    'n_clusters': 3,
    'algorithm': 'KMeans',
    'description': 'Anomaly detection using distance to nearest K-Means cluster center.'
}

with open(ARTIFACT_DIR / 'model_parameters.json', 'w', encoding='utf-8') as f:
    json.dump(model_parameters, f, indent=4)

print("Saved files:")
print("- artifacts/kmeans_model.pkl")
print("- artifacts/scaler.pkl")
print("- artifacts/model_parameters.json")

## Notes

This notebook is designed for educational and portfolio use. It demonstrates the logic of anomaly-based intrusion detection with K-Means clustering and can be extended with live packet capture or a backend API in future versions.